In [1]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import pandas as pd
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
import pickle
import ast
from time import time
import hashlib
import math
from tqdm import tqdm


In [2]:
def create_id(seed: str) -> str:
    return hashlib.md5(seed.encode()).hexdigest()

In [3]:
class MyEmbeddingFunction(EmbeddingFunction):
    def __init__(self, embedder):
        self.embedder = embedder
    def __call__(self, input: Documents) -> Embeddings:
        return self.embedder.embed_documents(input)

In [4]:
# !!! BELOW TO CHANGE !!! 
DATA_NAME = 'natural_questions'
DB_VERSION = 'v3'

EMBEDDING_MODEL_PATH = f'../models/multilingual-e5-small'
MODEL_KWARGS = {'device': 'cuda'}
ENCODE_KWARGS = {'normalize_embeddings': True, 'prompt': 'passage: '}
CHROMA_KWARGS = {"hnsw:space": "ip"}
FILTERING_OPTIONS = {"max_documents": 2000}
# !!! ABOVE TO CHANGE !!!

LOAD_DIR = f'../data/{DATA_NAME}'
DATASET_PATH = f'{LOAD_DIR}/natural_qa_chunked/chunked_nqa1.csv'
SAVE_DIR = f"../data/{DATA_NAME}/dbs/{DB_VERSION}"
DENSE_DB_SAVE_PATH = f'{SAVE_DIR}/densedb'
DB_LOG_PATH = f'{SAVE_DIR}/operation_info.json' 

#### Preparing

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_PATH,
    model_kwargs=MODEL_KWARGS,
    encode_kwargs=ENCODE_KWARGS 
)
ef = MyEmbeddingFunction(embeddings)

In [ ]:
client = chromadb.PersistentClient(path=DENSE_DB_SAVE_PATH)
collection = client.get_or_create_collection(name=DATA_NAME,  metadata=CHROMA_KWARGS, 
                                             embedding_function=ef)
print(collection.count())

In [7]:
df = pd.read_csv(DATASET_PATH)

In [ ]:
filtered_df = df.drop_duplicates(subset=['chunk'])
print(filtered_df.shape)

In [ ]:
filtered_df2 = filtered_df[filtered_df['index'] < FILTERING_OPTIONS['max_documents']]
print(filtered_df2.shape)

#### Vectorizing 

In [ ]:
documents = filtered_df2['chunk'].to_list()
metadatas = list(map(lambda idx_pair: {'document_index': idx_pair[1], 'chunk_index': idx_pair[0]}, enumerate(filtered_df2['index'].to_list())))
ids = list(map(lambda doc: create_id(doc), documents))

print(len(documents), len(metadatas), len(ids))

BATCH_SIZE = 1024

In [ ]:
vectorize_t_start = time()

for i in tqdm(range(math.floor(len(documents) // BATCH_SIZE))):
    collection.add(
        documents=documents[i*BATCH_SIZE:(i+1)*BATCH_SIZE],
        metadatas=metadatas[i*BATCH_SIZE:(i+1)*BATCH_SIZE],
        ids=ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE])

VECTORIZE_ELAPSED_TIME = round(time() - vectorize_t_start, 5)

In [ ]:
collection.count()

#### Saving Log

In [13]:
with open(DB_LOG_PATH, 'w') as fd:
    fd.write(json.dumps({
        "data_name": DATA_NAME,
        "db_version": DB_VERSION, "model_name": EMBEDDING_MODEL_PATH,
        "encode_kwargs": ENCODE_KWARGS, "chroma_kwargs": CHROMA_KWARGS,
        'filtering_options': FILTERING_OPTIONS,
        "vectorize_elapsed_sec_time": VECTORIZE_ELAPSED_TIME}, indent=1))